## Pattern Analysis - ISC and RSA

Given that each subject watched posts in a different order, we analyze the data post by post.
This allows us to see how brain responses vary for each specific post across subjects, across post type.

**Analysis plan inspiration from:**
> Chen, J., Leong, Y., Honey, C. et al. Shared memories reveal shared structure in neural activity across individuals. Nat Neurosci 20, 115–125 (2017). https://doi.org/10.1038/nn.4450

**Analysis Steps:**
1. Extract searchlight patterns from all subjects
2. Align patterns across subjects
3. Compute ISC or RSA for each searchlight sphere (mean or vector)
4. Statistical testing with permutations

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import importlib

from yy_fmri_kit import event_isc, isc
importlib.reload(event_isc)
importlib.reload(isc)

from yy_fmri_kit.event_isc.alignment import PatternAligner
from yy_fmri_kit.event_isc.extraction import SearchlightPatternExtractor
from yy_fmri_kit.static.event_isc.config import (
    ExtractionConfig,
    AlignmentConfig,
)
from yy_fmri_kit.event_isc.extraction import (SearchlightPatternExtractor)
from yy_fmri_kit.event_isc.alignment import (PatternAligner)
from yy_fmri_kit.isc.analyzer import ISCAnalyzer

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Data paths
DATA_DIR = Path("/path/to//data/derivatives/denoised")
EVENTS_CSV = Path("/path/to//behavioral_analyses/data/combined_events_with_bids.csv")
# Output directory
OUTPUT_DIR = Path("/path/to//data/derivatives/searchlight/GM/RSA")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Experiment parameters
TR = 1.0              # Your TR in seconds
SHIFT_TR = 4          # Hemodynamic delay in TRs
SEARCHLIGHT_RADIUS = 9.0  # Radius in mm

# Run types to analyze
RUN_TYPES = ["AntiLeft", "AntiRight", "ProLeft", "ProRight"]
TEST_SUBJECTS = [
    'sub-1', 'sub-6', 'sub-20', 'sub-21', 'sub-22', 'sub-23', 
    'sub-24', 'sub-25', 'sub-26', 'sub-27', 'sub-30']

In [ ]:
# ============================================================================
# OPTIONAL - CREATE A STANDARD MASK (if you don't have one)
# ============================================================================

from nilearn.datasets import load_mni152_brain_mask

standard_mask = load_mni152_brain_mask(resolution=2)
MASK_FILE = OUTPUT_DIR / "group_mask.nii.gz"
standard_mask.to_filename(str(MASK_FILE))
print(f"Saved mask: {MASK_FILE}, voxels: {int(standard_mask.get_fdata().sum())}")

print("Standard mask shape:", standard_mask.shape)  # should be (91, 109, 91)
print("N voxels:", int(standard_mask.get_fdata().sum()))  # should be ~130,000

In [ ]:
# ============================================================================
# OPTIONAL - CREATE A GRAY MATTER MASK (if you want to restrict to GM)
# ============================================================================
from nilearn.datasets import fetch_icbm152_2009
from nilearn import image
import nibabel as nib

# Reload GM probability map
icbm = fetch_icbm152_2009()
gm_prob = image.load_img(icbm['gm'])

# Reload BOLD reference
bold_files = list(DATA_DIR.rglob("*task-*_bold.nii.gz"))
ref_img = nib.load(str(bold_files[0]))
ref_3d = image.index_img(ref_img, 0)

# Resample and threshold
gm_resampled = image.resample_to_img(gm_prob, ref_3d, interpolation='linear')
gm_mask = image.math_img("img > 0.5", img=gm_resampled)

MASK_FILE = OUTPUT_DIR / "gm_mask_bold_space.nii.gz"
gm_mask.to_filename(str(MASK_FILE))

print("Shape:", gm_mask.shape)    # should be (91, 109, 91)
print("Voxels:", int(gm_mask.get_fdata().sum()))  # should be ~50-70k

In [ ]:
# ============================================================================
# STEP 0: LOAD DATA AND BUILD RUNS DICTIONARY
# ============================================================================

print("\n" + "="*80)
print("STEP 0: LOADING DATA")
print("="*80)

# Load events
print(f"\nLoading events from: {EVENTS_CSV}")
events_df = pd.read_csv(EVENTS_CSV)
print(f"✓ Loaded {len(events_df)} events")
print(f"  Subjects: {events_df['subject'].nunique()}")
print(f"  Run types: {events_df['run_type'].unique().tolist() if 'run_type' in events_df.columns else 'N/A'}")

# Build runs dictionary
print("\nBuilding runs dictionary...")
runs_dict = {}

# Find subject directories
subject_dirs = sorted([d for d in DATA_DIR.glob("sub-*") if d.is_dir()])
print(f"Found {len(subject_dirs)} subject directories")

for sub in TEST_SUBJECTS:  # Limit to first 3 subjects for testing
    subject_dir = DATA_DIR / sub
    
    # Find all NIfTI files for this subject
    nifti_files = list(subject_dir.rglob("*_bold.nii.gz"))
    
    # Filter for task runs (adjust pattern to match your files)
    task_files = [
        f for f in nifti_files 
        if any(task in f.name.lower() for task in ['antileft', 'antiright', 'proleft', 'proright'])
    ]
    
    if task_files:
        runs_dict[sub] = task_files
        print(f"  {sub}: {len(task_files)} runs")

print(f"\n✓ Built runs dictionary with {len(runs_dict)} subjects")
total_runs = sum(len(files) for files in runs_dict.values())
print(f"  Total runs: {total_runs}")

## ISC Searchlight Extraction and Analysis

In [ ]:
# ============================================================================
# ISC CONFIGURATION
# ============================================================================

MASK_FILE = OUTPUT_DIR / "gm_mask_bold_space.nii.gz"  # Path to group mask, or None for whole brain

# ISC parameters
N_PERMUTATIONS = 1000  # For statistical testing (1000+ recommended)
ALPHA = 0.05
CORRECTION = "fdr_bh"  # "fdr_bh", "bonferroni", or "none"

# Processing
N_JOBS = -1  # Number of parallel jobs (-1 = all CPUs)

print("="*80)
print("SEARCHLIGHT ISC PIPELINE")
print("="*80)
print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nConfiguration:")
print(f"  Data directory: {DATA_DIR}")
print(f"  Events file: {EVENTS_CSV}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  TR: {TR}s")
print(f"  Shift: {SHIFT_TR} TRs ({SHIFT_TR * TR}s)")
print(f"  Searchlight radius: {SEARCHLIGHT_RADIUS}mm")
print(f"  Run types: {RUN_TYPES}")
print(f"  Permutations: {N_PERMUTATIONS}")
print(f"  Parallel jobs: {N_JOBS}")

In [ ]:
# ============================================================================
# STEP 1: EXTRACT SEARCHLIGHT PATTERNS (MEANS)
# ============================================================================

print("\n" + "="*80)
print("STEP 1: EXTRACTING SEARCHLIGHT PATTERNS")
print("="*80)

# Configure extraction
extraction_config = ExtractionConfig(
    tr=TR,
    shift_tr=SHIFT_TR,
    time_unit="seconds",
    smoothing_fwhm=None,  # No smoothing for searchlight (done at searchlight level)
    detrend=True,
    standardize=False,    # ISC computation handles standardization
    searchlight_radius=SEARCHLIGHT_RADIUS,
    searchlight_n_jobs=N_JOBS,
    valid_run_types=RUN_TYPES,
    # IMPORTANT: Set actual column names here (must match events CSV)
    post_col="post_id",
    subject_col="bids_id",
    run_col="run",
)

print("\nExtraction configuration:")
print(f"  TR: {extraction_config.tr}s")
print(f"  Hemodynamic shift: {extraction_config.shift_tr} TRs")
print(f"  Searchlight radius: {extraction_config.searchlight_radius}mm")
print(f"  Post column: '{extraction_config.post_col}'")
print(f"  Subject column: '{extraction_config.subject_col}'")
print(f"  Run column: '{extraction_config.run_col}'")

# Create extractor
extractor = SearchlightPatternExtractor(extraction_config)

# Output directory for extracted patterns
extraction_output = OUTPUT_DIR / "searchlight_patterns"

print(f"\nExtracting patterns...")
print(f"  Output: {extraction_output}")

# Run batch extraction
summary = extractor.batch_extract(
    runs_dict=runs_dict,
    events_df=events_df,
    output_dir=extraction_output,
    mask_path=MASK_FILE,
    verbose=False,  # Set True for detailed logging
    aggregation = "mean",  # "mean", "median", or "vector"

)

print("\n✓ Extraction complete!")
print("\nExtraction summary:")
print(summary.groupby('status').size())

# Save summary
summary_file = OUTPUT_DIR / "extraction_summary.csv"
summary.to_csv(summary_file, index=False)
print(f"\nSaved summary to: {summary_file}")

# Show successful extractions
success = summary[summary['status'] == 'success']
if len(success) > 0:
    print(f"\nSuccessful extractions:")
    print(f"  Runs: {len(success)}")
    print(f"  Mean posts/run: {success['n_posts'].mean():.1f}")
    print(f"  Mean searchlights: {success['n_searchlights'].mean():.0f}")

# Show failures
failures = summary[summary['status'] != 'success']
if len(failures) > 0:
    print(f"\n⚠️  {len(failures)} runs failed:")
    print(failures[['subject', 'run_type', 'status']])


In [ ]:
# ============================================================================
# STEP 2: ALIGN PATTERNS ACROSS SUBJECTS
# ============================================================================

print("\n" + "="*80)
print("STEP 2: ALIGNING PATTERNS ACROSS SUBJECTS")
print("="*80)

# Configure alignment
alignment_config = AlignmentConfig(
    strategy="intersection",  # Only posts seen by ALL subjects
    min_subjects=2,
    allow_missing_posts=False,
    check_feature_dims=True,
)

print("\nAlignment configuration:")
print(f"  Strategy: {alignment_config.strategy}")
print(f"  Min subjects: {alignment_config.min_subjects}")

# Create aligner
aligner = PatternAligner(alignment_config)

# Align all run types
print("\nAligning run types...")
extraction_output = OUTPUT_DIR / "searchlight_patterns"
aligned_results = aligner.align_all_runs(
    output_dir=extraction_output,
    run_types=RUN_TYPES,
    pattern_type="searchlight"
)

print(f"\n✓ Aligned {len(aligned_results)} run types")

# Save aligned data
aligned_dir = OUTPUT_DIR / "aligned"
aligned_dir.mkdir(exist_ok=True, parents=True)

for run_type, result in aligned_results.items():
    output_file = aligned_dir / f"{run_type}_searchlight_aligned.npz"
    aligner.save_aligned(result, output_file, run_type)
    
    print(f"\n{run_type}:")
    print(f"  Subjects: {len(result['subjects'])}")
    print(f"  Posts: {len(result['post_ids'])}")
    print(f"  Searchlights: {result['data'][0].shape[1]}")
    print(f"  Shape per subject: {result['data'][0].shape}")
    print(f"  Saved: {output_file.name}")

In [ ]:
# ============================================================================
# STEP 3: COMPUTE ISC FOR EACH RUN TYPE
# ============================================================================

print("\n" + "="*80)
print("STEP 3: COMPUTING ISC WITH PERMUTATION TESTING")
print("="*80)

# Configure ISC analysis
analyzer = ISCAnalyzer(
    backend="native",  # Use your compute_isc
    fisher_z=True,     # Recommended for inference
    nan_policy="omit", # Handle missing data
)

print("\nISC configuration:")
print(f"  Backend: {analyzer.backend}")
print(f"  Fisher z: {analyzer.fisher_z}")
print(f"  Permutations: {N_PERMUTATIONS}")
print(f"  Alpha: {ALPHA}")
print(f"  Correction: {CORRECTION}")

# Analyze each run type
isc_results = {}
isc_output = OUTPUT_DIR / "isc_results"
isc_output.mkdir(exist_ok=True, parents=True)

for run_type in aligned_results.keys():
    print(f"\n{'#'*80}")
    print(f"Analyzing: {run_type}")
    print(f"{'#'*80}")
    
    # Load aligned data
    aligned_file = aligned_dir / f"{run_type}_searchlight_aligned.npz"
    
    # Run ISC analysis
    results = analyzer.analyze_from_file(
        aligned_file,
        n_permutations=N_PERMUTATIONS,
        alpha=ALPHA,
        correction=CORRECTION,
        random_seed=42,
    )
    
    isc_results[run_type] = results
    
    # Save results
    results_file = isc_output / f"{run_type}_searchlight_isc.npz"
    analyzer.save_results(results, results_file)
    print(f"\nSaved results to: {results_file}")

In [ ]:
# ============================================================================
# STEP 4: SUMMARY AND COMPARISON
# ============================================================================

print("\n" + "="*80)
print("STEP 4: SUMMARY")
print("="*80)

# Compare across run types
comparison_data = []
for run_type, results in isc_results.items():
    comparison_data.append({
        'run_type': run_type,
        'n_subjects': results['isc_subjectwise'].shape[0],
        'n_posts': results['isc_subjectwise'].shape[1] if results['isc_subjectwise'].ndim > 2 else 1,
        'n_searchlights': len(results['isc_mean']),
        'mean_isc': results['isc_mean'].mean(),
        'std_isc': results['isc_mean'].std(),
        'n_significant': results['n_significant'],
        'pct_significant': 100 * results['n_significant'] / len(results['significant']),
        'mean_isc_sig': results['isc_mean'][results['significant']].mean() if results['n_significant'] > 0 else np.nan,
    })

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*80)
print("ISC COMPARISON ACROSS RUN TYPES")
print("="*80)
print(comparison_df.to_string(index=False))

# Save comparison
comparison_file = OUTPUT_DIR / 'isc_comparison.csv'
comparison_df.to_csv(comparison_file, index=False)
print(f"\n✓ Saved comparison to: {comparison_file}")

In [ ]:
# ============================================================================
# STEP 5: SAVE RESULTS AS NIfTI MAPS FOR VISUALIZATION
# ============================================================================
import nibabel as nib
import numpy as np
from nilearn import image

# Load your ISC results
for run_type, results in isc_results.items():
    # Load centers from matching run type, not just any file
    sample_file = list((OUTPUT_DIR / "searchlight_patterns").rglob(f"*task-{run_type}*desc-searchlight*.npz"))[0]
    with np.load(sample_file, allow_pickle=True) as meta:
        affine = meta['affine']
        centers = meta['searchlight_centers']

    print(f"Centers for {run_type}: {centers.shape}")
    print(f"ISC results: {len(results['isc_mean'])}")

    # Get the brain mask shape
    group_mask = nib.load(str(MASK_FILE))
    
    # --- Map 1: Mean ISC ---
    isc_map = np.zeros(group_mask.shape)
    for i in range(len(results['isc_mean'])):
        x, y, z = centers[i].astype(int)
        isc_map[x, y, z] = results['isc_mean'][i]
    
    isc_nii = nib.Nifti1Image(isc_map, affine)
    isc_nii.to_filename(str(OUTPUT_DIR / f"{run_type}_isc_mean.nii.gz"))

    # --- Map 2: Significance mask (1 = significant, 0 = not) ---
    sig_map = np.zeros(group_mask.shape)
    for i in range(len(results['significant'])):
        x, y, z = centers[i].astype(int)
        sig_map[x, y, z] = float(results['significant'][i])
    
    sig_nii = nib.Nifti1Image(sig_map, affine)
    sig_nii.to_filename(str(OUTPUT_DIR / f"{run_type}_significant_mask.nii.gz"))
    
    # --- Map 3: Thresholded ISC (only significant voxels) ---
    thresh_map = isc_map * sig_map
    thresh_nii = nib.Nifti1Image(thresh_map, affine)
    thresh_nii.to_filename(str(OUTPUT_DIR / f"{run_type}_isc_thresholded.nii.gz"))
     
    print(f"{run_type}: saved 3 NIfTI files")
    print(f"  ISC range: {results['isc_mean'].min():.3f} to {results['isc_mean'].max():.3f}")
    print(f"  Significant voxels: {results['significant'].sum()}")

In [ ]:
# ============================================================================
# OPTIONAL: VISUALIZE P VALUES ON BRAIN
# ============================================================================

from nilearn import plotting
import nibabel as nib
import numpy as np

isc_output = OUTPUT_DIR / "isc_results"

for results_file in sorted(isc_output.glob("*_searchlight_isc.npz")):
    run_type = results_file.name.replace("_searchlight_isc.npz", "")
    res = np.load(results_file, allow_pickle=True)
    
    # Load centers matching this specific run type
    sample_file = list((OUTPUT_DIR / "searchlight_patterns").rglob(f"*task-{run_type}*desc-searchlight*.npz"))[0]
    with np.load(sample_file, allow_pickle=True) as meta:
        affine = meta['affine']
        centers = meta['searchlight_centers']
    
    group_mask = nib.load(str(MASK_FILE))
    
    # Sanity check
    print(f"\nProcessing {run_type}...")
    print(f"  centers: {centers.shape[0]}, isc_mean: {len(res['isc_mean'])}")
    print(f"  p_values max: {res['p_values'].max():.4f}, min: {res['p_values'].min():.6f}")
    print(f"  significant: {res['significant'].sum()}")
    
    assert centers.shape[0] == len(res['isc_mean']), "Mismatch between centers and ISC results!"
    
    xs, ys, zs = centers[:, 0].astype(int), centers[:, 1].astype(int), centers[:, 2].astype(int)
    
    neg_log_p_map = np.zeros(group_mask.shape)
    isc_map = np.zeros(group_mask.shape)
    sig_map = np.zeros(group_mask.shape)
    
    p = np.clip(res['p_values'], 1e-10, 1)
    neg_log_p_map[xs, ys, zs] = -np.log10(p)
    isc_map[xs, ys, zs] = res['isc_mean']
    sig_map[xs, ys, zs] = res['significant'].astype(float)
    
    print(f"  neg_log_p_map max: {neg_log_p_map.max():.3f}, non-zero: {(neg_log_p_map > 0).sum()}")
    
    nib.Nifti1Image(neg_log_p_map, affine).to_filename(str(OUTPUT_DIR / f"{run_type}_neg_log_p.nii.gz"))
    nib.Nifti1Image(isc_map, affine).to_filename(str(OUTPUT_DIR / f"{run_type}_isc_mean.nii.gz"))
    nib.Nifti1Image(isc_map * sig_map, affine).to_filename(str(OUTPUT_DIR / f"{run_type}_isc_thresholded.nii.gz"))
    
    neg_log_p_nii = nib.Nifti1Image(neg_log_p_map, affine)
    html = plotting.view_img(
        neg_log_p_nii,
        bg_img="MNI152",
        cmap="hot",
        threshold=1.3,
        vmin=1.3,
        vmax=4,
        title=f"Significant ISC - {run_type} (-log10 p, FDR corrected)",
    )
    html.save_as_html(str(OUTPUT_DIR / f"{run_type}_pvalue_interactive.html"))
    print(f"  Saved: {run_type}")

In [ ]:

from nilearn import plotting
import nibabel as nib
import numpy as np

isc_output = OUTPUT_DIR / "isc_results"

for results_file in sorted(isc_output.glob("*_searchlight_isc.npz")):
    run_type = results_file.name.replace("_searchlight_isc.npz", "")
    res = np.load(results_file, allow_pickle=True)
    
    # Load centers matching this specific run type
    sample_file = list((OUTPUT_DIR / "searchlight_patterns").rglob(f"*task-{run_type}*desc-searchlight*.npz"))[0]
    with np.load(sample_file, allow_pickle=True) as meta:
        affine = meta['affine']
        centers = meta['searchlight_centers']
    
    group_mask = nib.load(str(MASK_FILE))
# Build p-value map (significant voxels only, colored by p-value 0-1)
    p_map = np.zeros(group_mask.shape)
    sig_only = res['p_values'].copy()
    sig_only[~res['significant']] = 0  # zero out non-significant
    xs, ys, zs = centers.T  # unpack center coordinates
    p_map[xs, ys, zs] = sig_only
    
    p_nii = nib.Nifti1Image(p_map, affine)
    
    html = plotting.view_img(
        p_nii,
        bg_img="MNI152",
        cmap="hot_r",      # reversed: small p = bright/hot
        threshold=0.001,   # just above zero to exclude non-significant
        vmin=0.0,
        vmax=0.05,         # range is 0 to alpha
        title=f"Significant ISC - {run_type} (p-value, FDR corrected)",
    )
    html.save_as_html(str(OUTPUT_DIR / f"{run_type}_pvalue_interactive.html"))

In [ ]:
# ============================================================================
# STEP 6: VISUALIZE RESULTS (OPTIONAL)
# ============================================================================

from nilearn import plotting
import nibabel as nib
import numpy as np

run_type = "ProLeft"  # Change to your run type

# Load your thresholded ISC map
isc_nii = nib.load(str(OUTPUT_DIR / f"{run_type}_isc_thresholded.nii.gz"))

# --- 1. Glass brain (good overview) ---
plotting.plot_glass_brain(
    isc_nii,
    colorbar=True,
    title=f"ISC - {run_type}",
    cmap="hot",
    vmin=0.05,
    vmax=0.4,
    plot_abs=False,
    display_mode="lyrz",
)
plotting.show()

# --- 2. Stat map on MNI (more anatomical) ---
plotting.plot_stat_map(
    isc_nii,
    colorbar=True,
    title=f"ISC - {run_type}",
    cmap="hot",
    vmax=0.4,
    threshold=0.05,  # only show voxels above this ISC value
    display_mode="z",
    cut_coords=10,   # number of axial slices
)
plotting.show()

# --- 3. Interactive HTML (great for exploration) ---
html = plotting.view_img(
    isc_nii,
    bg_img="MNI152",
    cmap="hot",
    threshold=0.05,
    title=f"ISC - {run_type}",
    vmax=0.4,
)
html.save_as_html(str(OUTPUT_DIR / f"{run_type}_isc_interactive.html"))
print("Saved interactive map — open in browser")

# visualize significant voxels only in an interactive map
html_sig = plotting.view_img(
    str(OUTPUT_DIR / f"{run_type}_isc_thresholded.nii.gz"),  # ISC values, non-sig = 0
    bg_img="MNI152",
    cmap="hot",
    threshold=0.01,  # just above zero to exclude non-significant voxels
    title=f"Significant ISC - {run_type}",
    vmin=0.1,
    vmax=0.5,
)
html_sig.save_as_html(str(OUTPUT_DIR / f"{run_type}_significant_interactive.html"))
print(f"Saved: {run_type}_significant_interactive.html — open in browser")


## Inter-subejct RSA Searchlight Extraction and Analysis

In [ ]:
# ============================================================================
# RSA CONFIGURATION
# ============================================================================

MASK_FILE = OUTPUT_DIR / "gm_mask_bold_space.nii.gz"

# RSA parameters
# N_PERMUTATIONS = 1000  # For statistical testing (1000+ recommended) --- TODO: decide which type of permutation is best for RSA
ALPHA = 0.05
CORRECTION = "fdr_bh"  # "fdr_bh", "bonferroni", or "none"

# Processing
N_JOBS = -1  # Number of parallel jobs (-1 = all CPUs)

print("="*80)
print("SEARCHLIGHT RSA PIPELINE")
print("="*80)
print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nConfiguration:")
print(f"  Data directory: {DATA_DIR}")
print(f"  Events file: {EVENTS_CSV}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  TR: {TR}s")
print(f"  Shift: {SHIFT_TR} TRs ({SHIFT_TR * TR}s)")
print(f"  Searchlight radius: {SEARCHLIGHT_RADIUS}mm")
print(f"  Run types: {RUN_TYPES}")
# print(f"  Permutations: {N_PERMUTATIONS}")
print(f"  Parallel jobs: {N_JOBS}")

In [ ]:
# ============================================================================
# STEP 1: EXTRACT SEARCHLIGHT PATTERNS
# ============================================================================
from yy_fmri_kit.event_isc.extraction import searchlight
importlib.reload(searchlight)
from yy_fmri_kit.event_isc.extraction.searchlight import SearchlightPatternExtractor

print("\n" + "="*80)
print("STEP 1: EXTRACTING SEARCHLIGHT PATTERNS")
print("="*80)

# Configure extraction
extraction_config = ExtractionConfig(
    tr=TR,
    shift_tr=SHIFT_TR,
    time_unit="seconds",
    smoothing_fwhm=None,  # No smoothing for searchlight (done at searchlight level)
    detrend=True,
    standardize=False,    # ISC computation handles standardization
    searchlight_radius=SEARCHLIGHT_RADIUS,
    searchlight_n_jobs=N_JOBS,
    valid_run_types=RUN_TYPES,
    # IMPORTANT: Set actual column names here (must match events CSV)
    post_col="post_id",
    subject_col="bids_id",
    run_col="run",
)

print("\nExtraction configuration:")
print(f"  TR: {extraction_config.tr}s")
print(f"  Hemodynamic shift: {extraction_config.shift_tr} TRs")
print(f"  Searchlight radius: {extraction_config.searchlight_radius}mm")
print(f"  Post column: '{extraction_config.post_col}'")
print(f"  Subject column: '{extraction_config.subject_col}'")
print(f"  Run column: '{extraction_config.run_col}'")

# Create extractor
extractor = SearchlightPatternExtractor(extraction_config)

# Output directory for extracted patterns
extraction_output = OUTPUT_DIR / "searchlight_patterns"

print(f"\nExtracting patterns...")
print(f"  Output: {extraction_output}")

# Run batch extraction
summary = extractor.batch_extract(
    runs_dict=runs_dict,
    events_df=events_df,
    output_dir=extraction_output,
    mask_path=MASK_FILE,
    verbose=False,  # Set True for detailed logging
    aggregation = "vector",  # "mean", "median", or "vector"

)

print("\n✓ Extraction complete!")
print("\nExtraction summary:")
print(summary.groupby('status').size())

# Save summary
summary_file = OUTPUT_DIR / "extraction_summary.csv"
summary.to_csv(summary_file, index=False)
print(f"\nSaved summary to: {summary_file}")

# Show successful extractions
success = summary[summary['status'] == 'success']
if len(success) > 0:
    print(f"\nSuccessful extractions:")
    print(f"  Runs: {len(success)}")
    print(f"  Mean posts/run: {success['n_posts'].mean():.1f}")
    print(f"  Mean searchlights: {success['n_searchlights'].mean():.0f}")

# Show failures
failures = summary[summary['status'] != 'success']
if len(failures) > 0:
    print(f"\n⚠️  {len(failures)} runs failed:")
    print(failures[['subject', 'run_type', 'status']])


In [ ]:
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import nibabel as nib
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr
from tqdm import tqdm

In [ ]:
# --- Setup ---
npz_dir = OUTPUT_DIR / "searchlight_patterns"
run_type = "AntiLeft"  # Change to your run type
subject_order = TEST_SUBJECTS  # Ensure this matches the subjects you extracted patterns for

# Build the dictionary
npz_paths = {
    sub: next((npz_dir / sub).glob(f"*{run_type}*_desc-searchlight_patterns.npz"))
    for sub in subject_order
}

# Inspect it before proceeding
print(npz_paths)

In [ ]:
# ===========================================================================
# STEP 2: COMPUTE NEURAL RDMs FOR EACH RUN TYPE
# ===========================================================================

# For RSA, we skip the alignment step and compute RDMs directly from the extracted patterns
# We will compute a neural RDM for each subject and post, then average across posts for each run type

# =============================================================================
# 1. LOAD PATTERNS AND BUILD NEURAL RDM
# =============================================================================

def load_subject_pattern(
    npz_path: Path,
    aggregation: str = "mean_posts"
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Load a single subject's searchlight patterns and aggregate across posts.

    Args:
        npz_path:    Path to .npz file from SearchlightPatternExtractor
        aggregation: How to collapse across posts.
                     "mean_posts" → mean BOLD across posts per center (recommended)
                     "concat_posts" → concatenate post vectors (not used here)

    Returns:
        subject_pattern : (n_centers, n_voxels_max) — one vector per searchlight center
        centers         : (n_centers, 3) voxel coordinates
        affine          : (4, 4) affine matrix
    """
    data = np.load(npz_path, allow_pickle=True)
    patterns = data["patterns"]          # (n_posts, n_centers, max_voxels)
    centers  = data["searchlight_centers"]
    affine   = data["affine"]

    if aggregation == "mean_posts":
        # NaN-safe mean across posts
        subject_pattern = np.nanmean(patterns, axis=0)  # (n_centers, max_voxels)
    else:
        raise ValueError(f"Unknown aggregation: {aggregation}")

    return subject_pattern, centers, affine



def build_neural_rdm(
    npz_paths: dict[str, Path],
    subject_order: list[str],
    distance_metric: str = "correlation",
    verbose: bool = True,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Build a subject × subject neural RDM at every searchlight center.

    Args:
        npz_paths      : {subject_id: npz_path}
        subject_order  : list of subject IDs defining the RDM row/col order
        distance_metric: passed to scipy cdist — "correlation" (1 - Pearson) recommended
        verbose        : show progress bar

    Returns:
        neural_rdm  : (n_centers, n_subjects, n_subjects)
        centers     : (n_centers, 3)
        affine      : (4, 4)
    """
    n_subjects = len(subject_order)

    # Load all subjects
    all_patterns = []
    centers = affine = None

    for sub in tqdm(subject_order, desc="Loading subjects", disable=not verbose):
        pattern, centers, affine = load_subject_pattern(npz_paths[sub])
        all_patterns.append(pattern)  # each: (n_centers, max_voxels)

    # Stack → (n_subjects, n_centers, max_voxels)
    all_patterns = np.stack(all_patterns, axis=0)
    n_centers = all_patterns.shape[1]

    neural_rdm = np.zeros((n_centers, n_subjects, n_subjects), dtype=np.float32)

    for c in tqdm(range(n_centers), desc="Building neural RDMs", disable=not verbose):
        sphere_data = all_patterns[:, c, :]  # (n_subjects, max_voxels)

        # Mask NaN columns (voxels not present in all subjects' spheres)
        valid_cols = ~np.any(np.isnan(sphere_data), axis=0)

        if valid_cols.sum() < 2:
            neural_rdm[c] = np.nan
            continue

        sphere_clean = sphere_data[:, valid_cols]  # (n_subjects, n_valid_voxels)
        neural_rdm[c] = cdist(sphere_clean, sphere_clean, metric=distance_metric)

    return neural_rdm, centers, affine

neural_rdm, centers, affine = build_neural_rdm(npz_paths, TEST_SUBJECTS)
print(f"Neural RDM shape: {neural_rdm.shape} (centers, subjects, subjects)")


In [ ]:
# visualize the neural RDM for the first searchlight center
import matplotlib.pyplot as plt
center_idx = 0
plt.imshow(neural_rdm[center_idx], cmap="viridis")
plt.colorbar(label="Distance")
plt.title(f"Neural RDM for center {center_idx} at {centers[center_idx]}")
plt.xlabel("Subject")
plt.ylabel("Subject")
plt.xticks(ticks=np.arange(len(subject_order)), labels=subject_order, rotation=90)
plt.yticks(ticks=np.arange(len(subject_order)), labels=subject_order)
plt.tight_layout()
plt.show()

# visualize the mean neural RDM across all centers
mean_rdm = np.nanmean(neural_rdm, axis=0)
plt.imshow(mean_rdm, cmap="viridis")
plt.colorbar(label="Distance")
plt.title("Mean Neural RDM Across All Searchlight Centers")
plt.xlabel("Subject")
plt.ylabel("Subject")
plt.xticks(ticks=np.arange(len(subject_order)), labels=subject_order, rotation=90)
plt.yticks(ticks=np.arange(len(subject_order)), labels=subject_order)
plt.tight_layout()
plt.show()

In [ ]:
# ===========================================================================
# STEP 3: COMPUTE BEHAVIORAL RDM
# ===========================================================================

def build_behavioral_rdm(
    behavioral_df: pd.DataFrame,
    subject_order: list[str],
    subject_col: str = "subject_code",
    measure_col: str = "measure",
    distance_metric: str = "absolute_difference",
) -> np.ndarray:
    """
    Build a subject × subject behavioral RDM.

    Args:
        behavioral_df  : DataFrame with subject_col and measure_col
        subject_order  : list of subject IDs (must match neural RDM order)
        subject_col    : column name for subject IDs
        measure_col    : column name for the behavioral measure
        distance_metric: "absolute_difference" or "euclidean" (same for scalar)

    Returns:
        behavioral_rdm : (n_subjects, n_subjects) symmetric distance matrix
    """
    df = behavioral_df.set_index(subject_col)

    scores = np.array([df.loc[sub, measure_col] for sub in subject_order], dtype=float)

    if distance_metric == "absolute_difference":
        behavioral_rdm = np.abs(scores[:, None] - scores[None, :])
    else:
        behavioral_rdm = cdist(scores[:, None], scores[:, None], metric=distance_metric)

    return behavioral_rdm

# load behavioral data frame and merge with subject data frame to combine political attitude scores with BIDS IDs
behavioral_df = pd.read_csv("/path/to//behavioral_analyses/data/political_attitude_q_08122025.csv")
behavioral_df["subject_code"] = behavioral_df["subject_code"].str.replace(
    r'YY_PL_0*(\d+)', r'YY_PL_\1', regex=True)
subject_df = pd.read_csv("/path/to//behavioral_analyses/data/behavioral_with_bids.csv")

merged_df = pd.merge(behavioral_df, subject_df, on="subject_code", how="inner")

# Fix the measure column in the DataFrame first
merged_df["camp_support"] = pd.to_numeric(merged_df["camp_support"], errors="coerce")

# Filter merged_df to only those subjects
merged_df_filtered = merged_df[merged_df["bids_id"].isin(TEST_SUBJECTS)]

# Then build the RDM
behavioral_rdm = build_behavioral_rdm(
    behavioral_df=merged_df_filtered,
    subject_order=TEST_SUBJECTS,  # same order as neural RDM
    subject_col="bids_id",
    measure_col="camp_support",   # just the column name as a string
    distance_metric="absolute_difference",
)

In [ ]:
# visualize behavioral RDM
plt.imshow(behavioral_rdm, cmap="viridis")
plt.colorbar(label="Distance")
plt.title("Behavioral RDM (Political Attitude Distance)")
plt.xlabel("Subject")
plt.ylabel("Subject")
plt.xticks(ticks=np.arange(len(TEST_SUBJECTS)), labels=TEST_SUBJECTS, rotation=90)
plt.yticks(ticks=np.arange(len(TEST_SUBJECTS)), labels=TEST_SUBJECTS)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 3. COMPUTE RSA
# =============================================================================

def _upper_triangle(rdm: np.ndarray) -> np.ndarray:
    """Return upper triangle (excluding diagonal) as a flat vector."""
    idx = np.triu_indices(rdm.shape[0], k=1)
    return rdm[idx]


def compute_rsa_map(
    neural_rdm: np.ndarray,          # (n_centers, n_subjects, n_subjects)
    behavioral_rdm: np.ndarray,       # (n_subjects, n_subjects)
    verbose: bool = True,
) -> np.ndarray:
    """
    Compute Spearman correlation between neural and behavioral RDM upper triangles
    at each searchlight center.

    Returns:
        rsa_map : (n_centers,) — Spearman r at each center
    """
    n_centers = neural_rdm.shape[0]
    rsa_map = np.full(n_centers, np.nan, dtype=np.float32)

    beh_vec = _upper_triangle(behavioral_rdm)

    for c in tqdm(range(n_centers), desc="Computing RSA", disable=not verbose):
        neu_rdm_c = neural_rdm[c]
        if np.any(np.isnan(neu_rdm_c)):
            continue
        neu_vec = _upper_triangle(neu_rdm_c)
        r, _ = spearmanr(neu_vec, beh_vec)
        rsa_map[c] = r

    return rsa_map

rsa_map = compute_rsa_map(neural_rdm, behavioral_rdm)
print(f"RSA map shape: {rsa_map.shape} (one r value per searchlight center)")

In [ ]:
# visualize RSA map as a histogram
import matplotlib.pyplot as plt
plt.hist(rsa_map[~np.isnan(rsa_map)], bins=30, color="skyblue", edgecolor="black")
plt.title("Distribution of RSA Correlations Across Searchlight Centers")
plt.xlabel("Spearman r")
plt.ylabel("Number of Centers")
plt.tight_layout()
plt.show()

In [ ]:
# save RDMs (behavioral and neural) and RSA map to disk
np.savez(OUTPUT_DIR / "behavioral_rdm.npz", behavioral_rdm=behavioral_rdm)
np.savez(OUTPUT_DIR / "neural_rdm.npz", neural_rdm=neural_rdm)
np.savez(OUTPUT_DIR / "rsa_map.npz", rsa_map=rsa_map, centers=centers, affine=affine)
print("Saved behavioral RDM, neural RDM, and RSA map to disk")

In [ ]:
# =============================================================================
# 4. PERMUTATION TEST
# =============================================================================

def permutation_test(
    neural_rdm: np.ndarray,      # (n_centers, n_subjects, n_subjects)
    behavioral_rdm: np.ndarray,  # (n_subjects, n_subjects)
    rsa_map: np.ndarray,         # (n_centers,) observed
    n_permutations: int = 5000,
    seed: int = 42,
    verbose: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Vectorized permutation test: shuffle subject labels in the neural RDM,
    keep behavioral fixed. All centers are processed simultaneously per permutation
    rather than looping over centers inside the permutation loop.

    Strategy:
        - Pre-extract all upper triangles → (n_centers, n_pairs)
        - Rank them once (for Spearman) → (n_centers, n_pairs)
        - Per permutation: permute subject indices, re-extract upper triangles,
          rank, then correlate against behavioral ranks via dot product
        - This reduces the inner loop from O(n_centers × n_perms) to O(n_perms)

    Returns:
        p_map    : (n_centers,) one-tailed p-values
        null_dist: (n_centers, n_permutations) null distribution of r values
    """
    from scipy.stats import rankdata

    rng        = np.random.default_rng(seed)
    n_centers  = neural_rdm.shape[0]
    n_subjects = neural_rdm.shape[1]
    triu_idx   = np.triu_indices(n_subjects, k=1)
    n_pairs    = len(triu_idx[0])

    # --- Pre-extract and rank neural upper triangles: (n_centers, n_pairs) ---
    neural_uppers = neural_rdm[:, triu_idx[0], triu_idx[1]]  # (n_centers, n_pairs)

    # Rank each center's vector (Spearman = Pearson on ranks)
    # Apply rankdata along pairs axis
    neural_ranks = np.apply_along_axis(rankdata, 1, neural_uppers)  # (n_centers, n_pairs)

    # Mark centers with NaN — skip them
    valid_mask = ~np.any(np.isnan(neural_uppers), axis=1)  # (n_centers,)

    # --- Rank behavioral vector once ---
    beh_vec   = behavioral_rdm[triu_idx[0], triu_idx[1]]   # (n_pairs,)
    beh_ranks = rankdata(beh_vec)                           # (n_pairs,)

    # Normalize ranks for fast Pearson (= Spearman) via dot product
    def _normalize(x):
        """Zero-mean, unit-norm along last axis."""
        x = x - x.mean(axis=-1, keepdims=True)
        norm = np.linalg.norm(x, axis=-1, keepdims=True)
        norm = np.where(norm == 0, 1, norm)
        return x / norm

    beh_norm = _normalize(beh_ranks[None, :])              # (1, n_pairs)

    null_dist = np.full((n_centers, n_permutations), np.nan, dtype=np.float32)

    for perm in tqdm(range(n_permutations), desc="Permutations", disable=not verbose):
        perm_idx = rng.permutation(n_subjects)

        # Permute all centers at once
        perm_uppers = neural_rdm[:, perm_idx[triu_idx[0]], perm_idx[triu_idx[1]]]  # (n_centers, n_pairs)
        perm_ranks  = np.apply_along_axis(rankdata, 1, perm_uppers)                 # (n_centers, n_pairs)
        perm_norm   = _normalize(perm_ranks)                                         # (n_centers, n_pairs)

        # Spearman r = dot product of normalized rank vectors
        r_vals = (perm_norm * beh_norm).sum(axis=1)  # (n_centers,)
        null_dist[valid_mask, perm] = r_vals[valid_mask].astype(np.float32)

    # One-tailed p-value: proportion of null >= observed
    p_map = np.mean(null_dist >= rsa_map[:, None], axis=1)
 
    return p_map, null_dist

p_map, null_dist = permutation_test(neural_rdm, behavioral_rdm, rsa_map, n_permutations=1000)
print(f"Permutation test complete! p_map shape: {p_map.shape}, null_dist shape: {null_dist.shape}")

In [ ]:
# save permutation null distribution and p-values
np.savez(OUTPUT_DIR / "rsa_permutation_results.npz", p_map=p_map, null_dist=null_dist)
print("Saved permutation test results to disk")

In [ ]:
# =============================================================================
# 5. PROJECT RSA MAP BACK TO BRAIN VOLUME
# =============================================================================

def rsa_map_to_nifti(
    rsa_map: np.ndarray,          # (n_centers,)
    centers: np.ndarray,          # (n_centers, 3) voxel coords
    affine: np.ndarray,           # (4, 4)
    p_map: Optional[np.ndarray] = None,   # (n_centers,) optional
    p_threshold: float = 0.05,
) -> tuple[nib.Nifti1Image, Optional[nib.Nifti1Image]]:
    """
    Project RSA values (and optionally thresholded p-map) back into brain volume.

    Returns:
        rsa_img      : NIfTI image of RSA r-values
        p_img        : NIfTI image of p-values (or None)
    """
    # Infer volume shape from max voxel coordinates
    shape = tuple(centers.max(axis=0).astype(int) + 1)

    rsa_vol = np.zeros(shape, dtype=np.float32)
    rsa_vol[centers[:, 0], centers[:, 1], centers[:, 2]] = rsa_map

    rsa_img = nib.Nifti1Image(rsa_vol, affine)

    p_img = None
    if p_map is not None:
        p_vol = np.ones(shape, dtype=np.float32)
        p_vol[centers[:, 0], centers[:, 1], centers[:, 2]] = p_map
        # Threshold: set non-significant voxels to 1
        sig_mask = p_map <= p_threshold
        rsa_vol_thresholded = np.zeros(shape, dtype=np.float32)
        rsa_vol_thresholded[
            centers[sig_mask, 0],
            centers[sig_mask, 1],
            centers[sig_mask, 2]
        ] = rsa_map[sig_mask]
        p_img = nib.Nifti1Image(rsa_vol_thresholded, affine)

    return rsa_img, p_img

rsa_img, p_img = rsa_map_to_nifti(rsa_map, centers, affine, p_map, p_threshold=0.1)
rsa_img.to_filename(str(OUTPUT_DIR / f"{run_type}_rsa_map.nii.gz"))
if p_img is not None:
    p_img.to_filename(str(OUTPUT_DIR / f"{run_type}_rsa_map_thresholded.nii.gz"))
print(f"Saved RSA map and thresholded RSA map to disk")

In [ ]:
# visualize RSA map on brain
from nilearn import plotting
plotting.plot_stat_map(
    rsa_img,
    title=f"RSA Map - {run_type}",
    cmap="coolwarm",
    threshold=0.1,
    display_mode="z",
    cut_coords=10,
)
plotting.show()

# create an interactive HTML visualization
html = plotting.view_img(
    rsa_img,
    bg_img="MNI152",
    cmap="coolwarm",
    threshold=0.1,
    title=f"RSA Map - {run_type}",
    vmax=0.5,
    vmin=-0.5,
)
html.save_as_html(str(OUTPUT_DIR / f"{run_type}_rsa_interactive.html"))
print(f"Saved interactive RSA map — open {run_type}_rsa_interactive.html in browser")

In [ ]:
from nilearn import plotting

# After you have p_map from permutation testing:
rsa_img, thresh_img = rsa_map_to_nifti(
    rsa_map, centers, affine, 
    p_map=p_map, 
    p_threshold=0.1  # includes p<0.05 and p<0.1
)

# Interactive HTML colored by RSA r-value, showing only significant voxels
html = plotting.view_img(
    thresh_img,          # thresholded image (non-sig voxels = 0)
    bg_img="MNI152",
    cmap="RdBu_r",       # diverging colormap: blue=negative, red=positive
    threshold=0.001,     # just above zero to exclude non-sig voxels
    vmin=-0.5,
    vmax=0.5,
    title=f"RSA - {run_type} (p<0.10, uncorrected)",
)
html.save_as_html(str(OUTPUT_DIR / f"{run_type}_rsa_interactive.html"))

## Parcellated ISC and RSA Analysis (stimulus level)

In [ ]:
# TODO: all function are simplified, need to refactor and optimize

In [ ]:
import importlib
from pathlib import Path
from yy_fmri_kit.event_isc.extraction import parcel
importlib.reload(parcel)

from yy_fmri_kit.event_isc.extraction.parcel import Config, load_timeseries, load_events, extract_post_patterns, compute_isc, compute_rsa, permutation_test, fdr_correct, results_to_dataframe

In [ ]:
cfg = Config(
    data_dir   = Path("/path/to/data/derivatives/parcellated"),
    events_csv = Path("/path/to/behavioral_analyses/data/combined_events_with_bids.csv"),
    subjects   = ['sub-1', 'sub-6', 'sub-20', 'sub-21', 'sub-22',
                  'sub-23', 'sub-24', 'sub-25', 'sub-26', 'sub-27', 'sub-30'],
    run_types  = ['AntiLeft', 'AntiRight', 'ProLeft', 'ProRight'],
    tr         = 1.0,
    shift_tr   = 4,
    subject_col  = 'bids_id',
    run_col      = 'run',
    post_col     = 'post_id',
    onset_col    = 'onset_s',
    duration_col = 'duration_s',
    # Fix the glob to match your actual file structure:
    tsv_glob = "{subject}/{subject}_*_task-{run_type}_*atlas-Schaefer2018*timeseries.tsv",
)


events_df      = load_events(cfg)
ts_dict        = load_timeseries(cfg)          # {(sub, run_type): DataFrame}
patterns       = extract_post_patterns(ts_dict, events_df, cfg)
# patterns: {run_type: {post_id: array (n_subjects, n_parcels)}}

isc_r, isc_p       = compute_isc(patterns['AntiLeft'], cfg)
rsa_r, rsa_p       = compute_rsa(patterns['AntiLeft'])

In [ ]:
# visualize ISC results
import matplotlib.pyplot as plt
plt.hist(isc_r, bins=30, color="skyblue", edgecolor="black")
plt.title("Distribution of ISC Correlations Across Parcels")
plt.xlabel("Spearman r")
plt.ylabel("Number of Parcels")
plt.tight_layout()
plt.show()

In [ ]:
# visualize RSA results
plt.hist(rsa_r, bins=30, color="salmon", edgecolor="black")
plt.title("Distribution of RSA Correlations Across Parcels")
plt.xlabel("Spearman r")
plt.ylabel("Number of Parcels")
plt.tight_layout()
plt.show()

In [ ]:
from yy_fmri_kit.visualization import pattern_analysis
importlib.reload(pattern_analysis)
from yy_fmri_kit.visualization.pattern_analysis import (
    plot_rdm, plot_rdm_comparison, plot_group_rdm,
    plot_condition_bar, plot_isc_parcels, plot_network_summary,
    plot_null_distribution, plot_subject_isc, plot_rsa_scatter,
)

In [ ]:
# One subject's RDM for one condition
plot_rdm(patterns['AntiLeft'], subject_idx=0, run_type='AntiLeft')

# All 4 conditions side by side for one subject
plot_rdm_comparison(patterns, subject_idx=0, subject_label='sub-1')

# Group-average RDM across all subjects
plot_group_rdm(patterns)

In [ ]:
# Mean r ± SEM across parcels, one bar per condition
plot_condition_bar(all_isc, title="ISC across conditions")

# Top 20 significant parcels for one condition
plot_isc_parcels(all_isc['AntiLeft'], run_type='AntiLeft', top_n=20)

# Aggregated by Schaefer network (Vis, SomMot, Default, etc.)
plot_network_summary(all_isc)

# Per-subject bar in one specific parcel (like Chen Fig 2c)
pcc_idx = parcel_names.index('7Networks_LH_Default_PCC_1')
plot_subject_isc(isc_subj, pcc_idx, cfg.subjects, run_type='AntiLeft')